# Introduction

**Concept** <br>
Fit a *single GEV distribution* using annual maxima from **pooled data**<br>

**Options**
- **Option1**<br>
    pool all data per location for global stationary and non-stationary analysis<br><br>
- **Option2**<br>
    pool all data per location and year for time-dependent stationary analysis<br>

---
**Additional Notes**
- Using sim_year as the actual year 

**Notes on next steps:**
- 8 models × 680 locations = 5,440 independent analyses >> parallelize code on model level

---
**!!! ToDo**
- make flow diagram of model
- parallelize the code
- CONTINUE add Confidence Intervals for Return Levels

# Import Libraries

In [2]:
import sys
import random
import time
from datetime import datetime
from glob import glob
import warnings

import xarray as xr
from IPython.display import Markdown, display
from pandas import DataFrame

import func_gev as gev
import func_preparation as dbf
import func_plotting as dbplt
import func_utils as ut

warnings.filterwarnings("ignore", category=FutureWarning)

# Settings

In [3]:
path = '../input/Annual_max_DCPP_20260112/'
path_export = '../output/gev_analysis/pooled/'

In [4]:
hindcast_start = 1960
hindcast_end = 2026

In [5]:
return_periods = [10, 25, 50, 100, 200]
plot_period_evolution = ['10-year', '50-year', '100-year']

In [6]:
dic_timing = {}
dic_notes_analysis = {}

In [7]:
colors = [
    '#53354DFF','#7D4F73FF','#B887ADFF','#CAA5C2FF','#DBC3D6FF','#F5F5F5FF','#99E3DDFF',
    '#66D4CCFF','#33C6BBFF','#008A80FF','#005C55FF'
    ]

In [8]:
print_msg = True

export_report=True
display_results = False
save_regression_summary = True

# Import data

In [9]:
dic_timing['import data'] = {}
dic_timing['import data']['start'] = datetime.now()

ls_files = [file for file in glob(path + '*.nc')]
ls_files

['../input/Annual_max_DCPP_20260112/Annual_max_MIROC6.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_MPI-ESM1-2-HR.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_HadGEM3-GC31-MM.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_MRI-ESM2-0.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_BCC-CSM2-MR.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_CMCC-CM2-SR5.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_CanESM5.nc',
 '../input/Annual_max_DCPP_20260112/Annual_max_NorCPM1.nc']

In [10]:
dic_data_per_model = dict()

for en, file in enumerate(ls_files):
    model_name = file.split('/')[-1].split('.nc')[0].split('_')[-1]
    
    print(f'Importing data from model {model_name} ({en+1}/{len(ls_files)})...')
    model_name, ds_model = dbf.import_data_from_file(file) 
    
    dic_data_per_model[model_name] = dict({'raw data': ds_model})

dic_timing['import data']['end'] = datetime.now()

Importing data from model MIROC6 (1/8)...
Importing data from model MPI-ESM1-2-HR (2/8)...
Importing data from model HadGEM3-GC31-MM (3/8)...
Importing data from model MRI-ESM2-0 (4/8)...
Importing data from model BCC-CSM2-MR (5/8)...
Importing data from model CMCC-CM2-SR5 (6/8)...
Importing data from model CanESM5 (7/8)...
Importing data from model NorCPM1 (8/8)...


# Data Preparation 

## BiasCorrection and Selection of valid data 
Note, in case the validity check fails, it will return an error. 

Code parallelization for speed up, using `joblib` - saving ~60% (from 3min30sec down to 1min22sec)


In [11]:
dic_timing['data correction'] = {}
dic_timing['data correction']['start'] = datetime.now()
dic_data_per_model = dbf.data_preparation(ls_files=ls_files, dic_data_per_model=dic_data_per_model)
dic_timing['data correction']['end'] = datetime.now()

In [12]:
display(Markdown("**Data Overview**"))
display(Markdown("**Model · model shape: samples (~sim_years) | ensemble members | valid locations**"))
dic_notes_analysis['data overview'] = ut.create_data_overview(dic_data_per_model,ls_files)

display(Markdown(f"Execution time · {dic_timing['data correction']['end'] - dic_timing['data correction']['start']}sec"))

**Data Overview**

**Model · model shape: samples (~sim_years) | ensemble members | valid locations**

MIROC6 · (680, 2, 7054)
MPI-ESM1-2-HR · (630, 2, 5808)
HadGEM3-GC31-MM · (680, 2, 7216)
MRI-ESM2-0 · (320, 2, 3547)
BCC-CSM2-MR · (630, 2, 5236)
CMCC-CM2-SR5 · (690, 2, 6038)
CanESM5 · (660, 2, 5782)
NorCPM1 · (630, 2, 4558)
Overall, data is available from
        	8 models,
        	3547-7216 locations (originally 11022)
        	320-690 samples per model
        	 - with 63-69 unique sim_years
        	 - between 1961-2029


Execution time · 0:01:13.315906sec

## Pool data per location cross models

from 8 models with up to 680 samples (sim_years) and two members and 3547-7216 locations, pool all data together and group per location
> Restructure multi-model ensemble by site <br>
> - from dic[model] -> shape: (sim_year, ensemble_member, location) <br>
> - to dic[location] -> shape: (sim_year, ensemble_member, model)


In [13]:
dic_timing['data pooling'] = {}
dic_timing['data pooling']['start'] = datetime.now()
ls_notes = []

# ------------------------------------------------------------------------------------------
da_list = []
for model_name, dic_model in dic_data_per_model.items():
    da = dic_model['valid data'] 

    da_loc = dbf.sites_to_location(da)
    da_loc = da_loc.expand_dims(model=[model_name])

    da_list.append(da_loc)

combined = xr.concat(da_list, dim="model", join="outer")
dic_timing['data pooling']['end'] = datetime.now()

display(Markdown(f"\n**Overall, the combined dataset has the following dimensions**"))
message = f"""
        Overview of combined dataset
        \tFinal dimensions: {combined.dims}
        \tShape: {combined.shape}
        \tNumber of models: {combined.model.size}
        \tNumber of locations: {combined.location.size}
        """.strip()
print(message)
ls_notes.append(message)

display(Markdown(f"\n**Create and Store summary of locations for which we have no data in either of the models**"))
missing_locations = dbf.create_summary_location_w_missing_data(
    dic_data_per_model=dic_data_per_model, combined=combined, 
    dir_export='/'.join(path_export.split('/')[:-3]) + '/exploration'
    )
ls_notes.append(f'{len(missing_locations)} locations without any valid data found!')

# ------------------------------------------------------------------------------------------
dic_notes_analysis['data pooling'] = ls_notes


**Overall, the combined dataset has the following dimensions**

Overview of combined dataset
        	Final dimensions: ('model', 'sample', 'member', 'location')
        	Shape: (8, 690, 2, 9589)
        	Number of models: 8
        	Number of locations: 9589



**Create and Store summary of locations for which we have no data in either of the models**

1433 locations without any valid data found!
Loading formatted geocoded file...
Map saved as ../output/exploration/map_missingValidData_3.5kmRadius.html. You can open it in your browser and interact with it.


### Validation Check

In [14]:
dic_timing['data validity check'] = {}
dic_timing['data validity check']['start'] = datetime.now()

# ------------------------------------------------------------------------------------------
list_model_labels = list(dic_data_per_model.keys())

model_ex = random.choice(range(len(list_model_labels)))
model_label = list_model_labels[model_ex]
loc_ex = random.choice(range(dic_data_per_model[model_label]['valid data'].shape[-1]))
print(f"For validity check, use randomly selected model {model_label} and site-id {loc_ex}...")

display(Markdown(f"\n**Overview Original dataArray**"))
data_for_model_for_location, lon_target, lat_target = ut.get_dataset_overview_for_model_at_location(
    dic_data=dic_data_per_model, model_label=model_label, site_id=loc_ex
    )

display(Markdown(f"\n**Overview Revised dataArray**"))
revised_dataset, lon_rev, lat_rev = ut.get_dataset_overview_for_model_at_location(
    dic_data=combined, model_nr=model_ex, lon=lon_target, lat=lat_target
    )

For validity check, use randomly selected model MIROC6 and site-id 3152...



**Overview Original dataArray**

Model MIROC6 (None) - location-ID 3152 
Full dataframe (680, 2) vs reduced (529, 2)
coordinates in original dataset lon|lat: 28.05619|36.81284



**Overview Revised dataArray**

Model None (0) - location-ID None 
Full dataframe (690, 2) vs reduced (529, 2)
coordinates in original dataset lon|lat: 28.05619|36.81284


/Users/silviazieger/Coding/Python/UniCat_SeaLevelExtremes/.venv/lib/python3.12/site-packages/xarray/core/duck_array_ops.py:258: RuntimeWarning: invalid value encountered in cast
  return data.astype(dtype, **kwargs)


In [15]:
if (
    lon_target == lon_rev
    and lat_target == lat_rev
    and data_for_model_for_location.dropna().equals(revised_dataset.dropna()
    )
):
    message = "Validity Check performed successfully"
else:
    message = "Validity Check failed"

dic_timing['data validity check']['end'] = datetime.now()
dic_notes_analysis['data pooling'].append(message + ' ' + str(dic_timing['data validity check']['end']))

# Workflow GEV - Generalized Extreme Value

## OPTION1
Using all data available per location - from all years and models. Currently, selecting a subset of 10 sites, but later will be applied to all locations.

#### Initial trial with subset of 10 locations

In [16]:
dic_timing['GEV approach 1'] = {}
dic_timing['GEV approach 1']['start'] = datetime.now()

# ------------------------------------------------------------------------------------------
list_sites = random.choices(range(combined.shape[-1]),k=10)

dic_data_per_location = {}
for loc_ex in list_sites: 
    data_at_location = combined[:,:,:, loc_ex].to_dataframe().dropna().reset_index()
    data_at_location = data_at_location.rename(columns={'annualMax':'storm_surge'})
    dic_data_per_location[loc_ex] = data_at_location

### Run Analysis

Note, the output is stored as 
- visuals → png
- tabular data (DataFrames) → Parquet
- other objects (dicts, strings, floats) → Pickle

File structure
```results/
├─ location_1/
│   ├─ df.parquet
│   ├─ metrics.pkl
│   └─ notes.pkl
├─ location_2/
│   ├─ df.parquet
│   ├─ metrics.pkl
│   └─ notes.pkl
...
```

In [17]:
orig_stdout, orig_stderr, fh, logger, log_path = ut.initialize_logger(
    f"LOGS_GEVAnalysis_pooled_{datetime.now():%Y%m%d_%H%M%S}.log"
    )

# ------------------------------------------------------------------------------------------
dic_timing['GEV approach 1']['analysis start'] = datetime.now()
ls_notes = []

print("\n" + "="*100)
print("STORM SURGE GEV ANALYSIS - POOLED APPROACH PER LOCATION")
print("="*100)

# ------------------------------------------------------------------------------------------
dic_prepared, notes = ut.prepare_pooled_data(
    dic_data=dic_data_per_location,
    hindcast_start=hindcast_start,
    hindcast_end=hindcast_end
)
ls_notes.append(notes)

# ------------------------------------------------------------------------------------------
results = {}
for loc_id, df_prepared in dic_prepared.items():
    print("\n" + "-"*70)
    print(f"Analyzing location id {loc_id} ...")

    lon_loc = df_prepared.lon.unique()[0]
    lat_loc = df_prepared.lat.unique()[0]

    print("\tLookup location info...")
    location_closest = dbf.add_location_labels(DataFrame([lon_loc, lat_loc], index=['lon', 'lat']).T)
    location_info = ' '.join(location_closest.values[0][2:])
    print(f"\t → Closest location identified: {location_info}")

    result, ls_warnings = gev.analyze_per_location(
        df_prepared, loc_id, lat_loc, lon_loc, location_info, return_periods
    )
    if any(ls_warnings):
        ls_notes.append(ls_warnings)
    
    if result is None:
        message = f"\t→ Warning! No valid GEV fit for location id {loc_id}. Skipping ..."
        print(message)
        ls_notes.append(message)
        continue
    
    if export_report and path_export: 
        export_path_site = ut.save_location_results(            
            location_id=loc_id, result_location=result, base_dir=path_export, 
            plot_period_evolution=plot_period_evolution, display_results=display_results
            )

    result['file_path_report'] = export_path_site
    results[loc_id] = result

# ------------------------------------------------------------------------------------------
dic_timing['GEV approach 1']['analysis end'] = datetime.now()

print("\n" + "="*100)
time_diff = dic_timing['GEV approach 1']['analysis end'] - dic_timing['GEV approach 1']['analysis start']
print(f"✓ ANALYSIS COMPLETED IN {time_diff}!")
print("="*100)

# ------------------------------------------------------------------------------------------
dic_notes_analysis['GEV pooled analysis'] = ls_notes
sys.stdout = orig_stdout
sys.stderr = orig_stderr
logger.removeHandler(fh)
fh.close()

10 locations require an execution time of ~ 30-50sec<br> 
Upscaling to 11022 locations, will result in an execution time of ~9-10hours!

## OPTION2
Re-run stationary GEV per year (for location parameter; scale and shape remain as globally defined)

In [18]:
# del results
try:
    results.keys()
    print('✓ continue with available dictionary')
    
except NameError:
    print('import data from files...')
    results = ut.import_results_from_files(path_export)

✓ continue with available dictionary


In [19]:
dic_timing['GEV approach 2'] = {}
dic_timing['GEV approach 2']['analysis start'] = datetime.now()

# ------------------------------------------------------------------------------------------
results_extended, ls_notes_analysis = gev.execute_and_store_stat_gev_per_year(
    results=results, store_results=True, return_periods=return_periods
    )

# ------------------------------------------------------------------------------------------
for key, outer_list in ls_notes_analysis.items():
    ls_notes_analysis[key] = [inner for inner in outer_list if inner]
dic_notes_analysis['annual_statGEV'] = ls_notes_analysis
dic_timing['GEV approach 2']['analysis end'] = datetime.now()

# ------------------------------------------------------------------------------------------
time_diff = dic_timing['GEV approach 2']['analysis end'] - dic_timing['GEV approach 2']['analysis start']
print(f"Execution time for computing GEV per year: {time_diff}sec")

Conducting stationary GEV for siteID 2534 (#1 out of 10) grouped per year...
	skipping Return Level Calculation, no stationary GEV for 1961.0...


/Users/silviazieger/Coding/Python/UniCat_SeaLevelExtremes/script/func_gev.py:375: RuntimeWarning: invalid value encountered in sqrt
  ci_lower = z_T - 1.96 * sqrt(var_z)
/Users/silviazieger/Coding/Python/UniCat_SeaLevelExtremes/script/func_gev.py:376: RuntimeWarning: invalid value encountered in sqrt
  ci_upper = z_T + 1.96 * sqrt(var_z)


	...2026 (#66 out of 66 years)
Done!

Conducting stationary GEV for siteID 3217 (#2 out of 10) grouped per year...


/Users/silviazieger/Coding/Python/UniCat_SeaLevelExtremes/script/func_gev.py:375: RuntimeWarning: invalid value encountered in sqrt
  ci_lower = z_T - 1.96 * sqrt(var_z)
/Users/silviazieger/Coding/Python/UniCat_SeaLevelExtremes/script/func_gev.py:376: RuntimeWarning: invalid value encountered in sqrt
  ci_upper = z_T + 1.96 * sqrt(var_z)


	...2026 (#66 out of 66 years)
Done!

Conducting stationary GEV for siteID 2887 (#3 out of 10) grouped per year...
	skipping Return Level Calculation, no stationary GEV for 1961.0...


...

	...2026 (#66 out of 66 years)
Done!

Conducting stationary GEV for siteID 7054 (#9 out of 10) grouped per year...
	skipping Return Level Calculation, no stationary GEV for 1961.0...
	skipping Return Level Calculation, no stationary GEV for 1962.0...


/Users/silviazieger/Coding/Python/UniCat_SeaLevelExtremes/script/func_gev.py:375: RuntimeWarning: invalid value encountered in sqrt
  ci_lower = z_T - 1.96 * sqrt(var_z)
/Users/silviazieger/Coding/Python/UniCat_SeaLevelExtremes/script/func_gev.py:376: RuntimeWarning: invalid value encountered in sqrt
  ci_upper = z_T + 1.96 * sqrt(var_z)


	...2026 (#66 out of 66 years)
Done!

Conducting stationary GEV for siteID 2300 (#10 out of 10) grouped per year...
	skipping Return Level Calculation, no stationary GEV for 1961.0...


/Users/silviazieger/Coding/Python/UniCat_SeaLevelExtremes/script/func_gev.py:375: RuntimeWarning: invalid value encountered in sqrt
  ci_lower = z_T - 1.96 * sqrt(var_z)
/Users/silviazieger/Coding/Python/UniCat_SeaLevelExtremes/script/func_gev.py:376: RuntimeWarning: invalid value encountered in sqrt
  ci_upper = z_T + 1.96 * sqrt(var_z)


	...2026 (#66 out of 66 years)
Done!

Execution time for computing GEV per year: 0:00:38.261024sec


Execution time for 10 locations ~ 40sec<br> 
Upscaling to 11022 locations, will result in an execution time of ~12.5hours!

### Regression of location parameter over years
including uncertainty given by n_obs

**NOTE**<br>
> centering the year parameter due to the following warning:<br>
*"The condition number is large, 2.37e+05. This might indicate that there are strong multicollinearity or other numerical problems."*

In [20]:
dic_timing['GEV approach 2']['regression start'] = datetime.now()

# ------------------------------------------------------------------------------------------
en = 0
for site_id, dic_location in results_extended.items():
    en+=1
    print(
        f"\nPlotting GEV μ trend analysis for location id {site_id} (#{en}/{len(results_extended.keys())})...", 
        end="\r"
        )
    
    df_stat_gev_per_year = dic_location['fit results']['gev_stationary']['analysis_per_year'].dropna()
    df = df_stat_gev_per_year.reset_index().rename(columns={'index': 'year'})

    global_statgev_shape = dic_location['fit results']['gev_stationary']['shape']
    global_statgev_scale = dic_location['fit results']['gev_stationary']['scale']

    [
        df, wls_delta, weights, y_pred, year_grid, year_mean
        ] = gev.weighted_least_square_regression_annual_location(global_statgev_scale, global_statgev_shape, df)
    results_extended[site_id]['WLSdelta'] = dict({'summary': wls_delta, 'weights': weights})
    
    fig = dbplt.plot_gev_mu_trend(
        df=df,
        weights=weights,
        year_grid=year_grid,
        year_mean=year_mean,
        y_pred=y_pred,
        wls_delta=wls_delta,
        nonstat_years=dic_location['data'].year.values.astype(int), 
        nonstat=dic_location['fit results']['gev_nonstationary'],
        display_results=display_results,
        colors_reg=['#333333FF', '#C88D35FF'],
        markers_color='#99E3DDFF'
        )
    
    if save_regression_summary and ('file location' in dic_location.keys() or 'file_path_report' in dic_location.keys()):
        if 'file location' in dic_location.keys():
            save_path = dic_location['file location']
        else:
            save_path = dic_location['file_path_report']
        with open(save_path + '/WLSdelta_summary.html', 'w') as f:
            f.write( wls_delta.summary().as_html())
        
        lat = str(dic_location['location info']['lat'].round(3))
        lon = str(dic_location['location info']['lon'].round(3))
        country = dic_location['location info']['description'].split(',')[-1].strip()  
        file_name = f"/GEVTrendAnalysis_location_{str(site_id)}_{country}_{lat}|{lon}.png"
        fig.savefig(save_path+file_name, dpi=300, bbox_inches='tight')

    else:
        print("\t skipping saving GEV μ trend analysis ...")

# ------------------------------------------------------------------------------------------
dic_timing['GEV approach 2']['regression end'] = datetime.now()
time_diff = dic_timing['GEV approach 2']['regression end'] - dic_timing['GEV approach 2']['regression start']
print(
    f"\nExecution time for computing regression analysis for annual stationary GEV and non-stationary GEV: "
    f"{time_diff}sec"
    )


Plotting GEV μ trend analysis for location id 2534 (#1/10)...
Plotting GEV μ trend analysis for location id 3217 (#2/10)...
Plotting GEV μ trend analysis for location id 2887 (#3/10)...
Plotting GEV μ trend analysis for location id 5272 (#4/10)...
Plotting GEV μ trend analysis for location id 4780 (#5/10)...
Plotting GEV μ trend analysis for location id 5001 (#6/10)...
Plotting GEV μ trend analysis for location id 1353 (#7/10)...
Plotting GEV μ trend analysis for location id 6938 (#8/10)...
Plotting GEV μ trend analysis for location id 7054 (#9/10)...
Plotting GEV μ trend analysis for location id 2300 (#10/10)...
Execution time for computing regression analysis for annual stationary GEV and non-stationary GEV: 0:00:06.597114sec


In [21]:
sorted(results_extended.keys())

[1353, 2300, 2534, 2887, 3217, 4780, 5001, 5272, 6938, 7054]

# Create high-level Summary

##  Store Run Notes and Warnings

In [22]:
ut.store_analysis_notes(dic_notes_analysis, path_export)

Full log written to ../output/gev_analysis/pooled/2026-02-03_225820_analysisNotes.txt
Full log written to ../output/gev_analysis/pooled/2026-02-03_225820_analysisNotes.txt


# Parameters Summary

In [ ]:
from pandas import concat

In [26]:
site_id = list(results.keys())[0]
site_id

2534

In [27]:
DataFrame([
    results[site_id]['location info']['lat'], 
    results[site_id]['location info']['lon'], 
    results[site_id]['location info']['description']
], index=['lat', 'lon', 'closest point identified']).T

,lat,lon,closest point identified
0,43.416662,-4.713838,Llanes Asturias ES


In [28]:
DataFrame([
    results[site_id]['hindcast period'][0], results[site_id]['hindcast period'][1]
    ], index=['start', 'end'], columns=['hindcast period']).T

,start,end
hindcast period,1961,2026


In [29]:
results[site_id]['fit results']['gev_nonstationary']

{'mu0': np.float64(0.24148905379971997),
 'mu1': np.float64(-0.0008371468235532943),
 'sigma': np.float64(0.04250246453051493),
 'xi': np.float64(-0.17054066847676808),
 'trend_in': 'location',
 'n_obs': 5595,
 'log_likelihood': np.float64(9477.176150284156),
 'aic': np.float64(-18946.352300568313),
 'bic': np.float64(-18919.833786085328),
 'years_mean': np.float64(1994.4899016979446),
 'years_std': np.float64(16.80364539498762),
 'CI': {'mu_pred': array([0.2431575 , 0.2431575 , 0.2431575 , ..., 0.23991924, 0.23991924,
         0.23991924], shape=(5595,)),
  'mu_lower': array([0.24058905, 0.24058905, 0.24058905, ..., 0.23746314, 0.23746314,
         0.23746314], shape=(5595,)),
  'mu_upper': array([0.24572595, 0.24572595, 0.24572595, ..., 0.24237534, 0.24237534,
         0.24237534], shape=(5595,))}}

In [30]:
results[site_id]['fit results']['gev_stationary']

{'shape': np.float64(-0.1705877433056088),
 'location': np.float64(0.24148719581325018),
 'scale': np.float64(0.04251810798327402),
 'n_obs': 5595,
 'log_likelihood': np.float64(9476.155037279677),
 'aic': np.float64(-18946.310074559355),
 'bic': np.float64(-18926.421188697117),
 'dist_type': 'Weibull (Type III)',
 'tail_behavior': 'Light (bounded)',
 'analysis_per_year':          shape  location     scale n_obs log_likelihood         aic  \
 1962 -0.609021  0.268946  0.031376    16      35.483734  -64.967468   
 1963   -0.2054  0.245791  0.043037    24      40.353718  -74.707436   
 1964 -0.087974   0.24667   0.03746    31      54.421638 -102.843277   
 1965 -0.616098  0.237998  0.044057    38       71.64615   -137.2923   
 1966 -0.886549  0.256651  0.051015    48      91.346475  -176.69295   
 ...        ...       ...       ...   ...            ...         ...   
 2022 -0.162681  0.243377   0.04115    66     113.094703 -220.189406   
 2023 -0.279281  0.235837  0.047295    58      94.

In [31]:
results[site_id]['model_comparison']

{'lr_statistic': np.float64(2.042226008958096),
 'df': 1,
 'p_value': np.float64(0.1529856289746101),
 'delta_aic': np.float64(-0.04222600895809592),
 'delta_bic': np.float64(6.5874026117890025),
 'decision': 'No strong evidence for non-stationarity',
 'recommendation': 'Use stationary model (simpler)'}

In [33]:
wls_params = concat([results[site_id]['WLSdelta']['summary'].params, results[site_id]['WLSdelta']['summary'].bse], axis=1)
wls_params.columns = ['value', 'std err']
wls_params

,value,std err
const,0.240149,0.001541
year,-0.000057,0.000092


In [34]:
results[site_id]['WLSdelta']['summary'].summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            WLS Regression Results                            
==============================================================================
Dep. Variable:               location   R-squared:                       0.006
Model:                            WLS   Adj. R-squared:                 -0.010
Method:                 Least Squares   F-statistic:                    0.3805
Date:                Tue, 03 Feb 2026   Prob (F-statistic):              0.540
Time:                        23:02:54   Log-Likelihood:                 192.23
No. Observations:                  65   AIC:                            -380.5
Df Residuals:                      63   BIC:                            -376.1
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.2401      0.002    155.875      0.000       0.237       0.243
year       -5.665e-05   9.18e-05     -0.617      0.540      -0.000       0.000
==============================================================================
Omnibus:                       43.098   Durbin-Watson:                   2.532
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              196.955
Skew:                          -1.791   Prob(JB):                     1.71e-43
Kurtosis:                      10.739   Cond. No.                         16.8
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [35]:
results[site_id]['return_levels']

{'stationary': {'10-year': {'return_level': np.float64(0.32094401927235994),
   'CI_lower': np.float64(0.3134651322651895),
   'CI_upper': np.float64(0.32842290627953036)},
  '25-year': {'return_level': np.float64(0.3463005741780134),
   'CI_lower': np.float64(0.3388560553965377),
   'CI_upper': np.float64(0.35374509295948914)},
  '50-year': {'return_level': np.float64(0.36263181307832554),
   'CI_lower': np.float64(0.35516562358638426),
   'CI_upper': np.float64(0.3700980025702668)},
  '100-year': {'return_level': np.float64(0.3770157276772595),
   'CI_lower': np.float64(0.369502399406514),
   'CI_upper': np.float64(0.38452905594800496)},
  '200-year': {'return_level': np.float64(0.3897406424451899),
   'CI_lower': np.float64(0.38216417991909396),
   'CI_upper': np.float64(0.39731710497128586)}},
 'nonstationary_start': {'year': 1961,
  'values': {'10-year': {'return_level': np.float64(0.32258902727231187),
    'CI_lower': np.float64(0.3161453533838355),
    'CI_upper': np.float64(0.3

# Next Steps

summarize the intercept, slopes, CIs for different approaches and store results in human-readable format

#### Potential Visualizations
- Maps of 100-year return levels along European coastline
- Difference maps: non-stationary minus stationary → climate change impact
- Probability exceedance curves for selected cities
- Histograms / density of return levels → compare regions
- Time series of non-stationary μ or return levels → show increasing trends

In [ ]:
site_id = 5943
result_location = results_extended[site_id]

result_location.keys()

In [ ]:
result_location['location info']

In [ ]:
result_location['hindcast period'] # years

In [ ]:
# GEV Stationary
result_location['fit results']['gev_stationary']['analysis_per_year']

In [ ]:
# GEV non-Stationary
result_location['fit results']['gev_nonstationary']

In [ ]:
result_location['model_comparison']

In [ ]:
result_location['return_levels']